# LearnMate — M5–M7 experiments
Runs on a free Colab **T4** (Runtime → Change runtime type → GPU). Checkpoints and caches are written to Google Drive so a disconnect never loses work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -b claude/wonderful-bohr-1idsaa https://github.com/KirtirajChaudhari/LearnMate.git || (cd LearnMate && git pull)
%cd /content/LearnMate

In [ ]:
!pip -q install -e '.[gpu,eval,dev]'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# persist large/expensive artifacts on Drive
import os
D='/content/drive/MyDrive/learnmate'
for sub in ['data_raw','data_processed','data_cache','models','results']:
    os.makedirs(f'$D/{sub}', exist_ok=True)
!mkdir -p data && ln -sfn $D/data_raw data/raw && ln -sfn $D/data_processed data/processed && ln -sfn $D/data_cache data/cache && ln -sfn $D/models models && ln -sfn $D/results results

In [ ]:
!test -d data/raw/d2l-en || git clone https://github.com/d2l-ai/d2l-en.git data/raw/d2l-en
!git -C data/raw/d2l-en checkout -q 23d7a5aecceee57d1292c56e90cce307f183bb0a
!python -m learnmate.ingest --check
!python scripts/build_squad.py

## 1. Indices and calibration-split runs (allowed before pre-registration)

In [ ]:
!python scripts/build_indices.py

In [ ]:
!python -m eval.run --split squad-cal --stages all

In [ ]:
!python -m eval.run --split course-dev --stages all
!python -m eval.run --split course-dev --condition S4 --stages all
!python -m eval.run --split course-cal --stages all

Judge course-cal outputs (blinded) before calibrating — see `annotation/GUIDELINES.md` §6.

In [ ]:
!python -m learnmate.annotation judge-sheet --run results/runs/course-cal__clean --out data/annotations/judge_cal.jsonl

In [ ]:
!python -m eval.calibrate --squad-cal results/runs/squad-cal__clean --course-cal results/runs/course-cal__clean --judgments data/annotations/judgments_cal.jsonl --course-dev-s4 results/runs/course-dev__S4

## 2. STOP — write `research/preregistration.md` (STATUS: FROZEN), commit and tag `prereg-v1` before continuing.

## 3. Locked test runs (executed once)

In [ ]:
for cond in ['clean','S1','S2','S3','S4']:
    !python -m eval.run --split course-test --condition {cond} --stages all --require-prereg
!python -m eval.run --split squad-test --stages all --require-prereg
!python -m eval.run --split course-test --stages evaluator --require-prereg

Judge test outputs blinded (GUIDELINES §6), merge, then run the registered analysis.

In [ ]:
!python -m eval.analyze --run results/runs/course-test__clean --judgments data/annotations/judgments_test.jsonl
!python -m eval.analyze --run results/runs/squad-test__clean

In [ ]:
for cond in ['S1','S2','S3','S4']:
    !python -m eval.analyze --run results/runs/course-test__{cond} --judgments data/annotations/judgments_test.jsonl

In [ ]:
!python scripts/make_figures.py